# Knowledge Graph Pipeline: Extraction + Cross-Book Completion

Pipeline dua-pass untuk membangun Knowledge Graph lintas buku (Biologi, Fisika, Kimia Kelas XII).
- **Pass-1** — Ekstraksi KG dari PDF + cross-book linking awal + expert validation (ADC boost)
- **Pass-2** — Cross-book completion via semantic ANN + graph-aware LLM classifier
- **Ingest** — Persistensi graf final ke Neo4j Aura

In [ ]:
import os, json, re, time, glob, subprocess, random
import statistics as st
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from collections import Counter, defaultdict

import numpy as np
from google import genai
from google.genai import types as gtypes
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase

## Konfigurasi

In [ ]:
# Secrets are read from a .env file (see .env.example). Never hardcode keys.
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError:
    pass
PROJECT_DIR = Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

BOOKS_CONFIG = [
    {"name": "Biologi Kelas XII", "pdf_siswa": "textbook/Biologi_BS_KLS_XII_Rev.pdf", "pdf_guru": "textbook/Biologi_BG_KLS_XII.pdf"},
    {"name": "Fisika Kelas XII",  "pdf_siswa": "textbook/Fisika_BS_KLS_XII.pdf",      "pdf_guru": "textbook/Fisika_BG_KLS_XII.pdf"},
    {"name": "Kimia Kelas XII",   "pdf_siswa": "textbook/Kimia_BS_KLS_XII.pdf",       "pdf_guru": "textbook/Kimia_BG_KLS_XII.pdf"},
]

GEMINI_API_KEY       = os.getenv("GEMINI_API_KEY")
LLM_MODEL_EXTRACT    = 'gemini-2.5-flash'
LLM_MODEL_CLASSIFY   = 'gemini-2.5-flash'
EMBEDDING_MODEL      = 'paraphrase-multilingual-mpnet-base-v2'
CHUNK_SIZE, CHUNK_OVERLAP = 800, 200

# -- Pass-2 tuning --
COS_THRESHOLD = 0.75
TOP_K         = 15
CONF_THRESHOLD_BY_TYPE = {'LINTAS_BUKU_BERKAITAN_DENGAN': 0.85, 'default': 0.7}

ALLOWED_TYPES = [
    'LINTAS_BUKU_SAMA_DENGAN', 'LINTAS_BUKU_APLIKASI_DARI',
    'LINTAS_BUKU_PRASYARAT_UNTUK', 'LINTAS_BUKU_MEMPERDALAM',
    'LINTAS_BUKU_BERKAITAN_DENGAN',
]
SYMMETRIC    = {'LINTAS_BUKU_SAMA_DENGAN', 'LINTAS_BUKU_BERKAITAN_DENGAN'}
ASYMMETRIC   = {'LINTAS_BUKU_APLIKASI_DARI', 'LINTAS_BUKU_PRASYARAT_UNTUK', 'LINTAS_BUKU_MEMPERDALAM'}
HIERARCHICAL = {'LINTAS_BUKU_PRASYARAT_UNTUK'}
NEIGHBORHOOD_MAX_EDGES = 5

# -- Neo4j --
NEO4J_URI  = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASS = os.getenv("NEO4J_PASS")
NEO4J_DB   = os.getenv("NEO4J_DB")

# -- Intra-book relation types --
INTRA_RELATION_TYPES = [
    'MENDEFINISIKAN', 'MENYEBABKAN', 'MEMUNGKINKAN', 'MENGATUR',
    'BAGIAN_DARI', 'BERINTERAKSI_DENGAN', 'BERGANTUNG_PADA', 'MEMPENGARUHI',
]

# -- Cross-book type definitions (shared by both passes) --
TYPE_DEFINITIONS = """
1. LINTAS_BUKU_SAMA_DENGAN     -- A dan B merujuk pada entitas/konsep yang sama di buku berbeda. Simetris.
2. LINTAS_BUKU_APLIKASI_DARI   -- A adalah penerapan/manifestasi konsep B di disiplin lain. Asimetris.
3. LINTAS_BUKU_PRASYARAT_UNTUK -- A adalah prasyarat untuk memahami B di buku lain. Asimetris, hierarkis.
4. LINTAS_BUKU_MEMPERDALAM     -- A memperdalam/memperluas pemahaman konsep B di buku lain. Asimetris.
5. LINTAS_BUKU_BERKAITAN_DENGAN -- Berkaitan secara umum lintas-buku (fallback). Simetris.
"""

client = genai.Client(api_key=GEMINI_API_KEY)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f'PROJECT_DIR : {PROJECT_DIR}')
# Sanity check: pastikan PDF ketemu.
for b in BOOKS_CONFIG:
    for k in ("pdf_siswa", "pdf_guru"):
        p = PROJECT_DIR / b[k]
        print(("  OK      " if p.exists() else "  MISSING ") + str(p))

print(f'\nLLM Extract : {LLM_MODEL_EXTRACT}')
print(f'LLM Classify: {LLM_MODEL_CLASSIFY}')
print(f'Embedding   : {EMBEDDING_MODEL}')
print(f'COS={COS_THRESHOLD} | TOP_K={TOP_K} | CONF={CONF_THRESHOLD_BY_TYPE}')


## Pass-1: Ekstraksi KG dari PDF

Langkah: Load PDF -> Extract TOC/chapter/glossary/materi pokok -> Assign nodes ke chapter -> Gemini extraction -> Dedup -> Save -> Expert validation + ADC boost.

### 1.1 Load PDF + Chunking

In [ ]:
all_books = {}

for cfg in BOOKS_CONFIG:
    name = cfg['name']
    # Path PDF relatif terhadap PROJECT_DIR (folder notebook), jadi jalan di mesin mana pun.
    pdf_siswa = PROJECT_DIR / cfg['pdf_siswa']
    pdf_guru = PROJECT_DIR / cfg['pdf_guru']
    docs   = SimpleDirectoryReader(input_files=[str(pdf_siswa)]).load_data()
    docs_g = SimpleDirectoryReader(input_files=[str(pdf_guru)]).load_data()
    parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    all_books[name] = {
        'documents': docs, 'documents_bg': docs_g,
        'nodes': parser.get_nodes_from_documents(docs),
        'nodes_bg': parser.get_nodes_from_documents(docs_g),
        'chapters': [], 'glossary': {}, 'materi_pokok_map': {},
        'chapter_nodes': {}, 'book_graph': None,
    }
    print(f'{name}: {len(all_books[name]["nodes"])} chunks (siswa), {len(all_books[name]["nodes_bg"])} chunks (guru)')

print(f'\nLoaded {len(all_books)} books.')

### 1.2 Extract TOC, Chapters, Glossary, Materi Pokok

In [ ]:
def extract_chapters_from_toc(toc_text):
    chapter_iter = list(re.finditer(
        r"(?i)(bab\s+(?:\d+|[IVXLCDM]+))\s+(.+?)\s*\.{2,}\s*(\d+)", toc_text))
    chapters = []
    for i, m in enumerate(chapter_iter):
        name = m.group(2).strip()
        page = int(m.group(3))
        if name.lower().startswith(('glosarium', 'indeks', 'daftar', 'rangkuman', 'asesmen', 'refleksi')):
            continue
        toc_start = m.end()
        toc_end = chapter_iter[i + 1].start() if i + 1 < len(chapter_iter) else len(toc_text)
        sub_matches = re.findall(
            r"(?m)^[ \t]*([A-Z])\.\s+(.+?)\s*\.{2,}\s*(\d+)",
            toc_text[toc_start:toc_end])
        subchapters = [{'label': l, 'name': n.strip(), 'page_start': int(p)} for l, n, p in sub_matches]
        chapters.append({'name': name, 'page_start': page, 'subchapters': subchapters})
    return sorted(chapters, key=lambda x: x['page_start'])


def extract_glossary(documents):
    """Ekstrak glosarium langsung dari PDF pakai PyMuPDF.
    Buku Kemdikbud punya 3 format glosarium berbeda, jadi dicoba 3 metode lalu
    dipilih yang menghasilkan istilah TERBANYAK:
      A. font bold  -> istilah=teks bold, definisi=teks regular   (Kimia, Fisika)
      B. block      -> baris-1 block = istilah, sisanya = definisi
      C. line+titik -> istilah=baris pendek, definisi=baris2 sampai diakhiri titik (Biologi)
    Glosarium yang membentang beberapa halaman ikut terbaca (sampai marker Indeks/Daftar Pustaka).
    """
    import fitz
    if not documents:
        return {}
    pdf_path = documents[0].metadata.get('file_path') or documents[0].metadata.get('file_name')
    if not pdf_path:
        return {}
    doc = fitz.open(pdf_path)
    n = doc.page_count
    starts = [i for i in range(n) if 'glosarium' in doc[i].get_text().lower() and i > n * 0.3]
    if not starts:
        return {}
    start = starts[0]
    gpages = [start]
    for p in range(start + 1, n):
        low = doc[p].get_text().lower()
        if any(s in low for s in ('indeks', 'daftar pustaka', 'daftar kredit', 'profil pelaku', 'biodata')):
            break
        gpages.append(p)

    def _method_bold():
        g, term, buf = {}, None, []
        def fl():
            nonlocal term, buf
            if term and buf:
                d = re.sub(r'\s+', ' ', ' '.join(buf)).strip().lstrip(':').strip()
                t = term.strip().rstrip(':').strip()
                if t and d and t.lower() != 'glosarium' and len(t) <= 60:
                    g.setdefault(t, d)
            buf = []
        for p in gpages:
            for b in doc[p].get_text('dict')['blocks']:
                if 'lines' not in b:
                    continue
                for ln in b['lines']:
                    for sp in ln['spans']:
                        txt = sp['text']
                        if not txt.strip():
                            continue
                        bold = bool(sp['flags'] & 2 ** 4) or 'bold' in sp['font'].lower()
                        big = sp['size'] >= 13
                        if big and txt.strip().lower() == 'glosarium':
                            continue
                        if bold and not big:
                            if buf:
                                fl(); term = txt
                            else:
                                term = (term + ' ' + txt) if term else txt
                        else:
                            if term:
                                buf.append(txt)
        fl()
        return g

    def _method_block():
        g = {}
        for p in gpages:
            for x0, y0, x1, y1, txt, *_ in doc[p].get_text('blocks'):
                lines = [l.strip() for l in txt.split('\n') if l.strip()]
                if len(lines) < 2:
                    continue
                if lines[0].lower() in ('glosarium', 'daftar pustaka') or lines[0].isdigit():
                    continue
                t = lines[0].rstrip(':').strip()
                d = re.sub(r'\s+', ' ', ' '.join(lines[1:])).strip()
                if t and d and len(t) <= 60:
                    g.setdefault(t, d)
        return g

    def _method_lineperiod():
        g, term, buf = {}, None, []
        def fl():
            nonlocal term, buf
            if term and buf:
                d = re.sub(r'\s+', ' ', ' '.join(buf)).strip()
                t = term.strip().rstrip(':').strip()
                if t and d and len(t) <= 45 and t.lower() != 'glosarium':
                    g.setdefault(t, d)
            buf = []
        for p in gpages:
            for line in doc[p].get_text().split('\n'):
                s = line.strip()
                if not s or s.isdigit() or s.lower() == 'glosarium':
                    continue
                if 'untuk SMA' in s or 'Kelas XII' in s:
                    continue
                if term is None:
                    term = s
                else:
                    buf.append(s)
                    if s.endswith('.'):
                        fl(); term = None
        fl()
        return g

    return max([_method_bold(), _method_block(), _method_lineperiod()], key=len)


def extract_materi_pokok(documents_bg, chapters):
    mp_map = {ch['name']: [] for ch in chapters}
    mp_text = ''
    for doc in documents_bg:
        low = doc.text.lower()
        if any(x in low for x in ['materi pokok', 'pokok materi', 'skema pembelajaran']):
            mp_text += doc.text + '\n'
    lines = [l.strip() for l in mp_text.split('\n') if l.strip()]
    for ch_name in mp_map:
        for line in lines:
            if (len(line.split()) < 10 and line[0:1].isupper()
                    and not any(s in line for s in ['Peserta didik', 'Tujuan', 'Kegiatan'])):
                if line not in mp_map[ch_name]:
                    mp_map[ch_name].append(line)
    return mp_map


def roman_to_int(roman):
    vals = {'i': 1, 'v': 5, 'x': 10, 'l': 50, 'c': 100, 'd': 500, 'm': 1000}
    total, prev = 0, 0
    for ch in reversed(roman.lower()):
        v = vals.get(ch, 0)
        total += -v if v < prev else v
        prev = v
    return total


def get_page_number(node):
    label = node.metadata.get('page_label', '')
    if label.isdigit():
        return int(label)
    try:
        return roman_to_int(label)
    except Exception:
        return 0


for book_name, bd in all_books.items():
    print(f'\n--- {book_name} ---')
    # TOC
    toc_text = ''
    for doc in bd['documents'][:20]:
        low = doc.text.lower()
        if ('daftar isi' in low or 'bab' in low) and re.search(r'\.{3,}\s*\d+', doc.text):
            toc_text += doc.text + '\n'
    chapters = extract_chapters_from_toc(toc_text)
    # Enrich prev/next
    for i, ch in enumerate(chapters):
        ch['previous'] = chapters[i-1]['name'] if i > 0 else None
        ch['next'] = chapters[i+1]['name'] if i < len(chapters)-1 else None
        ch['next_page'] = chapters[i+1]['page_start'] if i < len(chapters)-1 else 10000
    print(f'  Chapters: {len(chapters)}')
    # Glossary
    glossary = extract_glossary(bd['documents'])
    print(f'  Glossary: {len(glossary)} terms')
    # Materi pokok
    mp_map = extract_materi_pokok(bd['documents_bg'], chapters)
    # Assign nodes to chapters
    chapter_nodes = {}
    for node in bd['nodes']:
        page = get_page_number(node)
        for ch in chapters:
            if ch['page_start'] <= page < ch['next_page']:
                chapter_nodes.setdefault(ch['name'], []).append(node)
                break
    for ch_name, nodes in chapter_nodes.items():
        print(f'    {ch_name}: {len(nodes)} nodes')

    bd['chapters'] = chapters
    bd['glossary'] = glossary
    bd['materi_pokok_map'] = mp_map
    bd['chapter_nodes'] = chapter_nodes

print('\nPreprocessing complete.')

### 1.3 Extraction Functions

In [ ]:
def _normalize_text(s):
    s = (s or '').lower().strip()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s)


def _tokenize_meaningful(s):
    stopwords = {
        'dan', 'atau', 'yang', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan',
        'dalam', 'suatu', 'adalah', 'merupakan', 'sebagai', 'oleh', 'itu', 'ini',
        'bab', 'materi', 'konsep', 'proses', 'sistem', 'bagian', 'jenis', 'teori',
        'hukum', 'kelas', 'xii',
    }
    return {t for t in _normalize_text(s).split() if len(t) >= 4 and t not in stopwords}


def collect_other_books_context(all_books, current_book, limit_per_book=80):
    context = {}
    for bname, bd in all_books.items():
        if bname == current_book:
            continue
        cands = list(bd.get('glossary', {}).keys())[:40]
        for mp_list in bd.get('materi_pokok_map', {}).values():
            cands.extend(mp_list[:10])
        for ch in bd.get('chapters', []):
            cands.append(ch.get('name', ''))
            for sub in ch.get('subchapters', []):
                cands.append(sub.get('name', ''))
        seen, uniq = set(), []
        for c in cands:
            c = (c or '').strip()
            if c and c.lower() not in seen:
                seen.add(c.lower())
                uniq.append(c)
        context[bname] = uniq[:limit_per_book]
    return context


def infer_cross_links(concept_name, concept_desc, other_books_concepts, current_book, max_links=2):
    base_tokens = _tokenize_meaningful(f'{concept_name} {concept_desc}')
    if not base_tokens:
        return []
    scored = []
    for bname, cands in other_books_concepts.items():
        if bname == current_book:
            continue
        for c in cands:
            overlap = base_tokens & _tokenize_meaningful(c)
            if len(overlap) >= 2:
                scored.append((len(overlap), bname, c, overlap))
    scored.sort(key=lambda x: x[0], reverse=True)
    links, used = [], set()
    for _, bname, c, overlap in scored:
        k = (bname.lower(), c.lower())
        if k in used:
            continue
        used.add(k)
        links.append({
            'target_book': bname, 'target_concept': c,
            'relation_type': 'BERKAITAN_DENGAN',
            'explanation': f'Konsep berbagi kata kunci: {", ".join(sorted(overlap))}.',
        })
        if len(links) >= max_links:
            break
    return links


def extract_triplets(text_chunk, grade, chapter, subchapters=None, previous_chapter=None,
                     next_chapter=None, glossary=None, materi_pokok=None, other_books_concepts=None):
    sub_section = ''
    if subchapters:
        sub_list = '\n'.join(f'  {s["label"]}. {s["name"]}' for s in subchapters)
        sub_section = f'SUBCHAPTER:\n{sub_list}\nGunakan nama subchapter persis untuk field "name" di "subtopics".'

    mp_section = ''
    if materi_pokok:
        mp_list = '\n'.join(f'  {i+1}. {m}' for i, m in enumerate(materi_pokok[:25]))
        mp_section = f'MATERI POKOK (BUKU GURU):\n{mp_list}'

    glos_section = ''
    if glossary:
        glos_items = '\n'.join(f'  {t}: {d[:100]}' for t, d in list(glossary.items())[:25])
        glos_section = f'GLOSARIUM:\n{glos_items}'

    allowed_targets = list(other_books_concepts.keys()) if other_books_concepts else []
    cross_section = ''
    if other_books_concepts:
        for bname, concepts in other_books_concepts.items():
            if concepts:
                clist = '\n'.join(f'  - {c}' for c in concepts[:20])
                cross_section += f'KONSEP DARI {bname}:\n{clist}\n\n'

    prompt = f"""Kamu membangun knowledge graph dari 3 buku: Biologi, Fisika, Kimia Kelas XII.
HIERARKI: Kelas -> Bab -> Subchapter -> Konsep

Buku: {grade} | Bab: {chapter}
Bab sebelumnya: {previous_chapter or "Tidak ada"} | Bab berikutnya: {next_chapter or "Tidak ada"}

{sub_section}
{mp_section}
{glos_section}
{cross_section}

ATURAN CROSS-BOOK:
1. cross_book_links HARUS menghubungkan ke buku lain, BUKAN buku saat ini ({grade}).
2. target_book hanya boleh: {allowed_targets}
3. Isi 1-3 cross_book_links jika ada keterkaitan kuat, atau [] jika tidak ada.

OUTPUT FORMAT (STRICT JSON):
{{
  "chapter_summary": "Ringkasan 3-5 kalimat.",
  "subtopics": [{{
    "name": "...",
    "concepts": [{{
      "name": "...", "description": "Deskripsi 1-2 kalimat.",
      "glossary_validated": true/false, "materi_pokok_ref": "...",
      "cross_book_links": [{{
        "target_book": "...", "target_concept": "...",
        "relation_type": "SAMA_DENGAN|APLIKASI_DARI|PRASYARAT_UNTUK|MEMPERDALAM|BERKAITAN_DENGAN",
        "explanation": "..."
      }}],
      "relations": [{{
        "type": "MENDEFINISIKAN|MENYEBABKAN|MEMUNGKINKAN|MENGATUR|BAGIAN_DARI|BERINTERAKSI_DENGAN|BERGANTUNG_PADA|MEMPENGARUHI",
        "target": "...", "description": "..."
      }}]
    }}]
  }}],
  "chapter_relations": [{{
    "type": "PRASYARAT|MEMPERSIAPKAN", "target_chapter": "...",
    "description": "...", "concept_links": [{{"source_concept": "...", "target_concept": "...", "explanation": "..."}}]
  }}]
}}

Semua output Bahasa Indonesia.

Teks:
{text_chunk}"""

    response = client.models.generate_content(model=LLM_MODEL_EXTRACT, contents=prompt)
    text = response.text.replace('```json', '').replace('```', '').strip()
    parsed = json.loads(text)

    valid_books = set(allowed_targets)
    for sub in parsed.get('subtopics', []):
        for c in sub.get('concepts', []):
            raw = c.get('cross_book_links', []) or []
            cleaned = []
            for link in raw:
                tb = (link.get('target_book') or '').strip()
                tc = (link.get('target_concept') or '').strip()
                if not tb or tb == grade or (valid_books and tb not in valid_books) or not tc:
                    continue
                cleaned.append({
                    'target_book': tb, 'target_concept': tc,
                    'relation_type': (link.get('relation_type') or 'BERKAITAN_DENGAN').strip(),
                    'explanation': (link.get('explanation') or '').strip(),
                })
            if not cleaned and other_books_concepts:
                cleaned = infer_cross_links(c.get('name', ''), c.get('description', ''),
                                            other_books_concepts, grade)
            c['cross_book_links'] = cleaned
            c['relations'] = [
                {'type': r['type'].strip(), 'target': r['target'].strip(),
                 'description': r.get('description', '').strip()}
                for r in (c.get('relations') or [])
                if (r.get('type') or '').strip() and (r.get('target') or '').strip()
            ]
    return parsed

### 1.4 Run Extraction

In [ ]:
for book_name, bd in all_books.items():
    print(f'\n=== {book_name} ===')
    chapters = bd['chapters']
    chapter_nodes = bd['chapter_nodes']
    other_ctx = collect_other_books_context(all_books, book_name)
    for ob, oc in other_ctx.items():
        print(f'  Context from {ob}: {len(oc)} candidates')

    book_graph = {'grade': book_name, 'chapters': []}
    for ch in chapters:
        ch_name = ch['name']
        ch_text = '\n\n'.join(n.text for n in chapter_nodes.get(ch_name, []))
        if not ch_text.strip():
            print(f'  SKIP {ch_name}: no text')
            continue
        try:
            result = extract_triplets(
                ch_text, book_name, ch_name,
                subchapters=ch.get('subchapters', []),
                previous_chapter=ch['previous'], next_chapter=ch['next'],
                glossary=bd['glossary'],
                materi_pokok=bd['materi_pokok_map'].get(ch_name, []),
                other_books_concepts=other_ctx,
            )
            ch_result = {
                'chapter': ch_name,
                'chapter_summary': result.get('chapter_summary', ''),
                'previous': ch['previous'], 'next': ch['next'],
                'subchapters': [s['name'] for s in ch.get('subchapters', [])],
                'subtopics': result.get('subtopics', []),
                'chapter_relations': result.get('chapter_relations', []),
            }
            cross_n = sum(len(c.get('cross_book_links', []))
                          for st in ch_result['subtopics'] for c in st.get('concepts', []))
            concept_n = sum(len(st.get('concepts', [])) for st in ch_result['subtopics'])
            book_graph['chapters'].append(ch_result)
            print(f'  {ch_name}: {len(ch_result["subtopics"])} subtopics, {concept_n} concepts, {cross_n} cross-links')
        except Exception as e:
            print(f'  ERROR {ch_name}: {str(e)[:120]}')

    bd['book_graph'] = book_graph
    print(f'  Done: {len(book_graph["chapters"])} chapters')

print('\nExtraction complete.')

### 1.5 Dedup, Save, Expert Validation

In [ ]:
def _normalize_pair(s):
    return re.sub(r'\s+', ' ', (s or '').strip().lower())

def _pair_key(sb, sc, tb, tc):
    return tuple(sorted([(_normalize_pair(sb), _normalize_pair(sc)),
                         (_normalize_pair(tb), _normalize_pair(tc))]))

# -- Dedup --
seen_pairs = set()
kept, removed = 0, 0
for book_name, bd in all_books.items():
    bg = bd.get('book_graph')
    if not bg:
        continue
    for ch in bg.get('chapters', []):
        for sub in ch.get('subtopics', []):
            for c in sub.get('concepts', []):
                cleaned = []
                for link in c.get('cross_book_links', []) or []:
                    tb = (link.get('target_book') or '').strip()
                    tc = (link.get('target_concept') or '').strip()
                    if not tb or not tc or tb == book_name:
                        removed += 1
                        continue
                    pk = _pair_key(book_name, c['name'], tb, tc)
                    if pk in seen_pairs:
                        removed += 1
                        continue
                    seen_pairs.add(pk)
                    cleaned.append(link)
                    kept += 1
                c['cross_book_links'] = cleaned

print(f'Dedup: kept={kept}, removed={removed}, unique_pairs={len(seen_pairs)}')

# -- Save pre-boost --
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
for book_name, bd in all_books.items():
    bg = bd.get('book_graph')
    if not bg:
        continue
    fname = f'{book_name}_{timestamp}.json'
    fpath = OUTPUTS_DIR / fname
    fpath.write_text(json.dumps(bg, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved pre-boost: {fname}')

# -- Expert validation + ADC boost --
print('\nRunning expert validation + ADC boost...')
result = subprocess.run(
    ['python3', 'expert-validation/apply_expert_validation_adc.py', '--aggressive', '--min-relations', '3'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Expert validation failed.')

# -- Reload boosted JSON --
for book_name, bd in all_books.items():
    pattern = str(OUTPUTS_DIR / f'{book_name}_*_expert_boosted.json')
    candidates = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    if candidates:
        bd['book_graph'] = json.loads(Path(candidates[0]).read_text(encoding='utf-8'))
        print(f'Loaded boosted: {os.path.basename(candidates[0])}')
    else:
        print(f'WARNING: no boosted JSON for {book_name}')

print('\nPass-1 complete. all_books now contains boosted graphs.')

## Neo4j Ingest

Ingest graf ekstraksi Pass-1 ke Neo4j Aura dalam 3 pass: nodes, relasi dalam-buku, relasi cross-book. Edge completion Pass-2 (LINTAS_BUKU_*) ditambahkan setelah ini oleh handoff ke src/completion.py.

In [ ]:
def _safe_rel(rel_type):
    return re.sub(r'[^A-Z0-9_]', '_', rel_type.upper().replace(' ', '_'))

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
driver.verify_connectivity()
print(f'Connected to Neo4j: {NEO4J_URI}')

data_to_ingest = {}
for book_name, bd in all_books.items():
    bg = bd.get('book_graph')
    if bg:
        data_to_ingest[bg['grade']] = bg
        print(f'  {book_name}: {len(bg.get("chapters", []))} chapters')

# PASS 1: Nodes
print('\n--- Pass 1: Nodes ---')
with driver.session(database=NEO4J_DB) as s:
    for grade in data_to_ingest:
        s.run('MATCH (n) WHERE n.grade = $g DETACH DELETE n', g=grade)
        s.run('MATCH (g:Grade {name: $n}) DETACH DELETE g', n=grade)

    for grade, bg in data_to_ingest.items():
        s.run('MERGE (g:Grade {name: $n}) SET g.type = "KLS_XII"', n=grade)
        for ch in bg.get('chapters', []):
            ch_name = ch['chapter']
            s.run("""MERGE (c:Chapter {name: $name, grade: $grade})
                     SET c.summary = $summary
                     WITH c MATCH (g:Grade {name: $grade}) MERGE (g)-[:HAS_CHAPTER]->(c)""",
                 name=ch_name, grade=grade, summary=ch.get('chapter_summary', ''))
            if ch.get('next'):
                s.run("""MATCH (a:Chapter {name: $a, grade: $g})
                         MATCH (b:Chapter {name: $b, grade: $g})
                         MERGE (a)-[:NEXT_CHAPTER]->(b)""",
                     a=ch_name, b=ch['next'], g=grade)
            for sub in ch.get('subtopics', []):
                sub_name = sub['name']
                s.run("""MERGE (s:Subtopic {name: $name, chapter: $ch, grade: $g})
                         WITH s MATCH (c:Chapter {name: $ch, grade: $g}) MERGE (c)-[:HAS_SUBTOPIC]->(s)""",
                     name=sub_name, ch=ch_name, g=grade)
                for c in sub.get('concepts', []):
                    s.run("""MERGE (k:Concept {name: $name, grade: $g})
                             ON CREATE SET k.description=$desc, k.glossary_validated=$valid, k.materi_pokok_ref=$mp
                             ON MATCH  SET k.description=$desc, k.glossary_validated=$valid, k.materi_pokok_ref=$mp
                             WITH k MATCH (s:Subtopic {name: $sub, chapter: $ch, grade: $g})
                             MERGE (s)-[:HAS_CONCEPT]->(k)""",
                         name=c['name'], g=grade, desc=c.get('description', ''),
                         valid=c.get('glossary_validated', False), mp=c.get('materi_pokok_ref', ''),
                         sub=sub_name, ch=ch_name)
    print('Nodes ingested.')

# PASS 2: Intra-book relations
print('\n--- Pass 2: Intra-book Relations ---')
reuse_n, target_n, skip_n = 0, 0, 0
with driver.session(database=NEO4J_DB) as s:
    for grade, bg in data_to_ingest.items():
        existing = set()
        for ch in bg.get('chapters', []):
            for sub in ch.get('subtopics', []):
                for c in sub.get('concepts', []):
                    existing.add(c['name'])
        for ch in bg.get('chapters', []):
            for sub in ch.get('subtopics', []):
                for c in sub.get('concepts', []):
                    for rel in c.get('relations', []):
                        rt = (rel.get('type') or '').strip()
                        tgt = (rel.get('target') or '').strip()
                        if not rt or not tgt:
                            skip_n += 1
                            continue
                        rt_safe = _safe_rel(rt)
                        desc = rel.get('description', '')
                        if tgt in existing:
                            s.run(f'MATCH (k:Concept {{name: $src, grade: $g}}) '
                                  f'MATCH (c:Concept {{name: $tgt, grade: $g}}) '
                                  f'MERGE (k)-[r:{rt_safe}]->(c) SET r.description = $desc',
                                  src=c['name'], tgt=tgt, g=grade, desc=desc)
                            reuse_n += 1
                        else:
                            s.run('MERGE (t:ConceptTarget {name: $t, grade: $g})', t=tgt, g=grade)
                            s.run(f'MATCH (k:Concept {{name: $src, grade: $g}}) '
                                  f'MATCH (t:ConceptTarget {{name: $tgt, grade: $g}}) '
                                  f'MERGE (k)-[r:{rt_safe}]->(t) SET r.description = $desc',
                                  src=c['name'], tgt=tgt, g=grade, desc=desc)
                            target_n += 1
            for cr in ch.get('chapter_relations', []):
                rt_safe = _safe_rel(cr['type'])
                s.run(f'MATCH (a:Chapter {{name: $a, grade: $g}}) '
                      f'MATCH (b:Chapter {{name: $b, grade: $g}}) '
                      f'MERGE (a)-[r:{rt_safe}]->(b) SET r.description = $desc',
                      a=ch['chapter'], b=cr['target_chapter'], g=grade, desc=cr.get('description', ''))
    print(f'Intra-book: reuse={reuse_n}, target={target_n}, skipped={skip_n}')

# PASS 3: Cross-book relations
print('\n--- Pass 3: Cross-book Relations ---')
cross_n, skip_same, skip_unknown = 0, 0, 0
with driver.session(database=NEO4J_DB) as s:
    for grade, bg in data_to_ingest.items():
        for ch in bg.get('chapters', []):
            for sub in ch.get('subtopics', []):
                for c in sub.get('concepts', []):
                    for link in c.get('cross_book_links', []):
                        tb = (link.get('target_book') or '').strip()
                        tc = (link.get('target_concept') or '').strip()
                        if not tb or not tc:
                            continue
                        if tb == grade:
                            skip_same += 1
                            continue
                        if tb not in data_to_ingest:
                            skip_unknown += 1
                            continue
                        rt = link.get('relation_type', 'LINTAS_BUKU')
                        rt_safe = _safe_rel(f'LINTAS_BUKU_{rt}') if not rt.startswith('LINTAS_BUKU') else _safe_rel(rt)
                        desc = link.get('explanation', '')
                        s.run('MERGE (tc:Concept {name: $tgt, grade: $tg})', tgt=tc, tg=tb)
                        s.run(f'MATCH (src:Concept {{name: $src, grade: $sg}}) '
                              f'MATCH (tgt:Concept {{name: $tgt, grade: $tg}}) '
                              f'MERGE (src)-[r:{rt_safe}]->(tgt) SET r.description=$desc, r.relation_type=$rt_orig',
                              src=c['name'], sg=grade, tgt=tc, tg=tb, desc=desc, rt_orig=rt)
                        cross_n += 1
    print(f'Cross-book: {cross_n} links (skipped same={skip_same}, unknown={skip_unknown})')

# Verification
print('\n--- Verification ---')
with driver.session(database=NEO4J_DB) as s:
    for lbl in ['Grade', 'Chapter', 'Subtopic', 'Concept', 'ConceptTarget']:
        cnt = s.run(f'MATCH (n:{lbl}) RETURN count(n) AS c').single()['c']
        print(f'  {lbl}: {cnt}')
    total = s.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']
    inter = s.run('MATCH (a:Concept)-[r]->(b:Concept) WHERE a.grade <> b.grade RETURN count(r) AS c').single()['c']
    print(f'  Total relations: {total}')
    print(f'  Inter-book relations: {inter}')

driver.close()
print('\nNeo4j ingest complete.')

## Pass-2: Cross-Book Completion — handoff ke src/completion.py

Pass-2 **tidak** mengimplementasi ulang completion: ia memanggil fungsi nyata dari paket proyek (./src, salinan persis closure src/completion.py aplikasi Streamlit) terhadap graf Neo4j hasil Ingest di atas. Jadi listing ini adalah kode yang benar-benar menghasilkan hasil tesis (replication-faithful).

Metode: ANN via Neo4j vector index -> klasifikasi LLM graph-aware (vokab tertutup LINTAS_BUKU_* 5-tipe) -> gerbang confidence per-tipe -> tulis edge ke Neo4j + dump audit.

In [ ]:
# ── Pass-2 handoff to src/completion.py (the real pipeline code) ─────────────
# This section does NOT re-implement completion. It calls the exact functions
# from the project package (bundled in ./src next to this notebook) against the
# Neo4j graph populated by the Ingest section above. The attached ./src is a
# byte-for-byte copy of the Streamlit app's src/ completion closure, so this
# listing IS the code that produced the thesis results (replication-faithful).
#
# Requires: the ./src package beside this notebook + its deps
# (llama_index, neo4j, networkx, pydantic, tenacity, python-dotenv), and a
# populated Neo4j graph (run the Ingest section first). Creds come from .env.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # so `import src` resolves the bundled ./src

from src.completion import (
    build_embed_text,
    get_embeddings,
    find_similar_pairs_ann,
    classify_similar_pairs,
    dump_lintas_buku_results,
)
from src.schema_adapter import get_adapter

# Yhoga ontology: Concept{name, grade} nodes + closed LINTAS_BUKU_* vocab.
adapter = get_adapter('yhoga')

# Canonical Yhoga cross-book run (see thesis Bab 3 / lintas_buku_edges.t075_k15).
EMBED_MODEL = 'gemini/gemini-embedding-001'
CHAT_MODEL  = 'gemini/gemini-2.5-flash'
THRESHOLD, TOP_K, BATCH = 0.75, 15, 10

# Reuse the `driver` opened in the Ingest section above.
nodes = adapter.get_nodes_for_completion(driver)
print(f'Concepts pulled from Neo4j: {len(nodes)}')

texts = [build_embed_text(n, adapter=adapter) for n in nodes]
embeddings = get_embeddings(texts, model=EMBED_MODEL)

# Stage 1 — ANN candidate retrieval (Neo4j vector index, O(n log n)).
pairs = find_similar_pairs_ann(
    driver, nodes, embeddings, model=EMBED_MODEL,
    threshold=THRESHOLD, top_k=TOP_K, adapter=adapter,
    cross_grade_only=True,            # LINTAS_BUKU: only cross-Mata-Pelajaran pairs
    skip_existing_typed_edges=True,   # don't re-type already-connected pairs
)
print(f'ANN candidate pairs: {len(pairs)}')

# Stage 2 — graph-aware LLM relation typing (closed 5-type vocab, disk-cached).
classified = classify_similar_pairs(
    driver, pairs, llm_model=CHAT_MODEL, batch_size=BATCH, adapter=adapter,
)

# Stage 3 — persist accepted LINTAS_BUKU_* edges back to Neo4j.
written = 0
for item in classified:
    if item.get('rel_type') and item['rel_type'] != 'none':
        adapter.save_typed_relationship(
            driver,
            source=item['source'], target=item['target'],
            rel_type=item['rel_type'], confidence=item.get('confidence'),
            description=item.get('description', ''),
            source_grade=item.get('source_grade'),
            target_grade=item.get('target_grade'),
        )
        written += 1
print(f'Wrote {written} LINTAS_BUKU_* edges to Neo4j')

# Stage 4 — audit dump (friend-llm shape), mirrors the canonical reference JSON.
ts2 = datetime.now().strftime('%Y%m%d_%H%M%S')
dump_path = OUTPUTS_DIR / f'lintas_buku_edges.{ts2}.json'
dump_lintas_buku_results(
    classified, dump_path, driver=driver,
    params={
        'embed_model': EMBED_MODEL, 'chat_model': CHAT_MODEL,
        'threshold': THRESHOLD, 'top_k': TOP_K, 'scope': 'cross',
        'batch_size': BATCH,
    },
)
print(f'Audit dump: {dump_path.name}')


## Analisis

Confidence-by-type signature, distribusi arah PRASYARAT_UNTUK, dan bridge concepts.

In [ ]:
import statistics as st

# Real edges only (drop classifier "none").
edges = [e for e in classified if e.get('rel_type') and e['rel_type'] != 'none']

def short(g):
    return (g or '').replace(' Kelas XII', '')

print('=' * 60)
print('[1] CONFIDENCE-BY-TYPE SIGNATURE')
print('=' * 60)
by_type = defaultdict(list)
for e in edges:
    by_type[e['rel_type']].append(e.get('confidence', 0.0))
for t in sorted(by_type, key=lambda k: -st.mean(by_type[k])):
    cs = by_type[t]
    print(f'  {t:35s}  n={len(cs):3d}  mean={st.mean(cs):.2f}  '
          f'median={st.median(cs):.2f}  range=[{min(cs):.2f},{max(cs):.2f}]')

print()
print('=' * 60)
print('[2] DIRECTION DISTRIBUTION (PRASYARAT_UNTUK)')
print('=' * 60)
prereq = [e for e in edges if e['rel_type'] == 'LINTAS_BUKU_PRASYARAT_UNTUK']
dirs = Counter(f'{short(e.get("source_grade"))} -> {short(e.get("target_grade"))}' for e in prereq)
print(f'Total PRASYARAT_UNTUK: {len(prereq)}')
for d, n in dirs.most_common():
    print(f'  {d:30s}  {n:3d}  ({100*n/max(len(prereq),1):.0f}%)')

print()
print('=' * 60)
print('[3] BRIDGE CONCEPTS')
print('=' * 60)
concept_reaches = defaultdict(set)
for e in edges:
    concept_reaches[(e.get('source_grade'), e['source'])].add(short(e.get('target_grade')))
    concept_reaches[(e.get('target_grade'), e['target'])].add(short(e.get('source_grade')))
bridges = {k: v for k, v in concept_reaches.items() if len(v) > 1}
print(f'{len(bridges)} bridge concepts (touching 2+ subjects)')
for (grade, name), reaches in sorted(bridges.items(), key=lambda kv: -len(kv[1]))[:10]:
    print(f'  [{short(grade)}] {name!r} -> {sorted(reaches)}')

print()
print('=' * 60)
print('SUMMARY')
print('=' * 60)
type_counts = Counter(e['rel_type'] for e in edges)
print(f'Concepts (Neo4j)        : {len(nodes)}')
print(f'Cross-book pairs (ANN)  : {len(pairs)}')
print(f'Edges accepted          : {len(edges)}')
print(f'Edges rejected (none)   : {len(classified) - len(edges)}')
print(f'Type distribution       : {dict(type_counts)}')
unique_touched = len({(e.get('source_grade'), e['source']) for e in edges} |
                     {(e.get('target_grade'), e['target']) for e in edges})
print(f'Unique concepts touched : {unique_touched}/{len(nodes)} ({100*unique_touched/max(len(nodes),1):.1f}%)')
